# HUMEA ablations from the recorded reproduction source

Run a fresh matched baseline and loss removals from the exact historical tracked source archive. No strict replay or model corrections are applied.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile

MODE = "campaign"     # campaign | smoke | validate
EXPERIMENT = "db15k-20"
BATCH_SIZE = 512          # Paper setting; checkpointed for Kaggle T4
DOWNLOAD_DATA_IF_MISSING = True
WANDB_MODE = "online"   # requires the enabled WANDB_API_KEY Kaggle secret
WANDB_PROJECT = "humea-reproduction"
WANDB_ENTITY = "diencongle"
TOOL_ROOT = Path("/kaggle/working/HUMEA-tools")
WORK_ROOT = Path("/kaggle/working/HUMEA-reference")
INPUT_ROOT = Path("/kaggle/input")
DATA_URL = "https://drive.google.com/file/d/1a3aou1qe7Yzq6y_khyTq1La_kf3UI7sC/view?usp=drive_link"
SOURCE_REVISION = "d2751d64e17543a52692faf35c4170327777135f"  # Immutable reference-ablation tooling
CAMPAIGN = "db15k20-reference-ablations-20261024"
assert MODE in {"validate", "smoke", "campaign"}
print("Mode:", MODE, "Experiment:", EXPERIMENT, "Batch size:", BATCH_SIZE)

In [ ]:
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))
subprocess.run(["nvidia-smi"], check=False)

## Restore the exact recorded reproduction source

The public GitHub fork provides the runner and the original tracked source archive. Training runs from the verified archived source in its own directory. Existing attached Kaggle data is reused; API keys stay in Kaggle Secrets.


In [ ]:
if TOOL_ROOT.exists() or WORK_ROOT.exists():
    raise FileExistsError("Existing source/output directories found; start a fresh session or choose new directories")
subprocess.run(["git", "clone", "--quiet", "https://github.com/lecongdien2000/HUMEA.git", str(TOOL_ROOT)], check=True)
subprocess.run(["git", "checkout", "--quiet", SOURCE_REVISION], cwd=TOOL_ROOT, check=True)
actual_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=TOOL_ROOT, text=True).strip()
print("Orchestration commit:", actual_revision)
prepared = subprocess.check_output([sys.executable, str(TOOL_ROOT / "reference_source.py"),
                                   "--destination", str(WORK_ROOT)], text=True)
print(prepared)
import json
reference = json.loads(prepared)
assert reference["source_sha256"] == "545b95884f15dfc94ceae0d985cbe0968da5cc733f718ab611be9688e2e51b34"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_RUN_GROUP"] = CAMPAIGN
os.environ["WANDB_TAGS"] = "db15k20,reference-source,loss-ablation,seed42"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(WORK_ROOT / "requirements-kaggle.txt")], check=True)
# Never store an API key in the notebook or print it.
assert WANDB_MODE in {"online", "offline", "disabled"}
os.environ["WANDB_MODE"] = WANDB_MODE
os.environ["WANDB_PROJECT"] = WANDB_PROJECT
if WANDB_ENTITY:
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY
if WANDB_MODE == "online":
    if not WANDB_ENTITY:
        raise ValueError("Set WANDB_ENTITY to your W&B username/team")
    if not os.environ.get("WANDB_API_KEY"):
        from kaggle_secrets import UserSecretsClient
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    if not os.environ.get("WANDB_API_KEY"):
        raise ValueError("Enable the WANDB_API_KEY Kaggle secret before running")
print("W&B mode:", WANDB_MODE, "project:", WANDB_PROJECT)

if WANDB_MODE == "online":
    import wandb
    wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
print("W&B credential loaded; online run initialization is checked by the smoke jobs.")


## Locate or download the processed features

Attached, already-extracted `mmkb-datasets` is preferred. Otherwise the notebook extracts an attached `data.zip`; with Internet enabled it can download the authors' archive once.

In [ ]:
expected = WORK_ROOT / "data" / "mmkb-datasets"
if not expected.is_dir():
    extracted_candidates = [p for p in INPUT_ROOT.rglob("mmkb-datasets") if p.is_dir()]
    if extracted_candidates:
        data_link = WORK_ROOT / "data"
        if data_link.exists() or data_link.is_symlink():
            if data_link.is_dir() and not data_link.is_symlink():
                shutil.rmtree(data_link)
            else:
                data_link.unlink()
        data_link.symlink_to(extracted_candidates[0].parent, target_is_directory=True)
        print("Using attached extracted data:", extracted_candidates[0])
    else:
        archives = list(INPUT_ROOT.rglob("data.zip"))
        archive = archives[0] if archives else Path("/kaggle/working/humea-data.zip")
        if not archives:
            if not DOWNLOAD_DATA_IF_MISSING:
                raise FileNotFoundError("No attached data.zip and downloading is disabled")
            subprocess.run(["gdown", "--fuzzy", DATA_URL, "-O", str(archive)], check=True)
        (WORK_ROOT / "data").mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as data_zip:
            data_zip.extractall(WORK_ROOT / "data")
        print("Extracted", archive)
assert expected.is_dir(), f"Expected dataset directory was not created: {expected}"
print("Dataset root:", expected)

In [ ]:
base_command = [
    sys.executable, str(TOOL_ROOT / "kaggle_runner.py"), "--repo-root", str(WORK_ROOT),
    "--batch-size", str(BATCH_SIZE),
]
validate = subprocess.run(base_command + ["validate"], cwd=WORK_ROOT)
if validate.returncode != 0:
    raise RuntimeError("Preflight validation failed; review the missing-file output above")

In [ ]:
import time, signal
artifacts = WORK_ROOT / "artifacts"
artifacts.mkdir(exist_ok=True)
(artifacts / "reference-provenance.json").write_text(json.dumps({
    **reference, "tooling_revision": actual_revision, "strict_reproducibility": False,
    "baseline_and_ablations": [0, 7, 8]}, indent=2))
started = time.monotonic()
# Stop within 5.5 hours of training, leaving quota headroom for setup/output.
deadline = started + 5.5 * 3600

def save_evidence():
    if artifacts.is_dir():
        shutil.make_archive("/kaggle/working/HUMEA-results", "zip", WORK_ROOT, "artifacts")

def run_job(without, smoke):
    kind = "smoke" if smoke else "single"
    command = base_command + ["--without", str(without), kind]
    if not smoke:
        command += ["--experiment", "db15k-20"]
    console_log = Path("/kaggle/working") / f"HUMEA-{kind}-without{without}.log"
    remaining = deadline - time.monotonic()
    if remaining < 120:
        raise RuntimeError("Campaign time budget exhausted; available evidence has been saved")
    print("Starting", kind, "without", without, "console:", console_log, flush=True)
    try:
        with console_log.open("w") as stream:
            process = subprocess.Popen(command, cwd=WORK_ROOT, stdout=stream,
                stderr=subprocess.STDOUT, start_new_session=True)
            try:
                returncode = process.wait(timeout=remaining)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
                raise RuntimeError("Campaign time budget exhausted; stopped training process group")
        if returncode:
            print(console_log.read_text()[-6000:])
            raise RuntimeError(f"{kind} without {without} failed; stopping campaign")
    finally:
        save_evidence()
    manifest = json.loads((artifacts / "manifest.json").read_text())
    record_id = ("smoke-" if smoke else "") + "db15k-20" + (f"-without{without}" if without else "")
    record = manifest["experiments"][record_id]
    if record.get("source_sha256") != reference["source_sha256"]:
        raise RuntimeError("Tracked training source differs from the historical reference; stopping")
    print("Completed", kind, "without", without, "verified source:", record["source_sha256"], flush=True)

try:
    if MODE != "validate":
        for without in (0, 7, 8):
            run_job(without, smoke=True)
        if MODE == "campaign":
            for without in (0, 7, 8):
                run_job(without, smoke=False)
finally:
    save_evidence()


In [ ]:
import pandas as pd
summary_path = WORK_ROOT / "artifacts" / "summary.csv"
if summary_path.is_file():
    display(pd.read_csv(summary_path))
    archive_path = shutil.make_archive(
        "/kaggle/working/HUMEA-results", "zip", WORK_ROOT, "artifacts"
    )
    print("Saved results, W&B records, source snapshots and environments:", archive_path)
    if WANDB_MODE == "offline":
        print("Offline W&B records must be synced to W&B and shared publicly before submission.")
else:
    print("Validation finished; no training summary was created.")
